
# UNSW-NB15 — Memory‑Safe Feature Engineering (No scikit‑learn)

This notebook:
- Loads & merges the 4 parts with headers from `NUSW-NB15_features.csv`
- **Labels:** maps attack → **1**, benign/normal → **0**
- Drops duplicates; normalizes timestamps; creates simple ratios
- **Memory‑safe encoding:** drops huge‑cardinality IDs, caps categoricals (Top‑K), one‑hots only selected columns
- Downcasts to **float32** and scales numerics (standard or minmax)
- Saves **summary statistics**, **correlation heatmap**, **PCA scatter**, and a **cleaned CSV**


In [ ]:

from pathlib import Path

# ===== Config =====
DATA_DIR = Path(".")  # change to your dataset folder if needed
OUT_DIR = Path("artifacts_unsw_nb15_ipynb")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_FILE = DATA_DIR / "NUSW-NB15_features.csv"
PARTS = [DATA_DIR / f"UNSW-NB15_{i}.csv" for i in [1,2,3,4]]

# Columns commonly present
LABEL_CANDIDATES = ["label", "attack_cat", "class", "category", "target"]
TS_CANDIDATES = ["timestamp", "stime"]
FORCE_CAT = ["proto", "service", "state"]

# Choices
SCALER = "standard"   # "standard" or "minmax"
PCA_KEEP_VAR = 0.95   # keep 95% variance
TOPK_CATS = 50        # cap levels per categorical

print("Data directory:", DATA_DIR.resolve())
print("Features exists:", FEATURES_FILE.exists())
print("Parts exist:", [p.exists() for p in PARTS])


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def read_features_csv(path: Path):
    for enc in ("cp1252","latin1","utf-8","unicode_escape"):
        try:
            feats = pd.read_csv(path, header=None, encoding=enc)
            return feats[1].astype(str).str.strip().tolist()
        except Exception:
            pass
    raise RuntimeError(f"Could not read features from {path}")

def map_attack_benign(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip().str.lower()
    return s.apply(lambda v: 1 if v in ("attack","malicious","1","true") else 0).astype(int)

def add_ratio(df: pd.DataFrame, a: str, b: str, name: str):
    if a in df.columns and b in df.columns:
        den = pd.to_numeric(df[b], errors="coerce").replace(0, np.nan)
        num = pd.to_numeric(df[a], errors="coerce")
        df[name] = (num / den).fillna(0.0)

def cap_categories(s: pd.Series, k: int = 50) -> pd.Series:
    s = s.astype(str)
    top = s.value_counts(dropna=False).nlargest(k).index
    return s.where(s.isin(top), other="__other__")

def pca_numpy(X: np.ndarray, keep_var: float = 0.95):
    Xc = X - X.mean(axis=0, keepdims=True)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    explained = (S**2) / (Xc.shape[0] - 1)
    ratio = explained / explained.sum()
    k = int(np.searchsorted(np.cumsum(ratio), keep_var) + 1)
    X_pca = Xc @ Vt.T[:, :k]
    return X_pca, k


In [ ]:

# ===== Load headers & merge 4 parts =====
base_cols = read_features_csv(FEATURES_FILE)

sample = pd.read_csv(PARTS[0], header=None, nrows=1, low_memory=False)
raw_cols = sample.shape[1]

if raw_cols == len(base_cols) + 2:
    col_names = base_cols + ["label","attack_cat"]
elif raw_cols == len(base_cols) + 1:
    col_names = base_cols + ["label"]
else:
    col_names = base_cols

df = pd.concat([pd.read_csv(p, header=None, names=col_names, low_memory=False) for p in PARTS],
               ignore_index=True)
print("Merged shape:", df.shape)
df.head(3)


In [ ]:

# ===== Clean + label mapping + timestamps + ratios =====
df = df.drop_duplicates().copy()
df.columns = df.columns.astype(str).str.strip()

lower_to_orig = {c.lower(): c for c in df.columns}
label_key = next((nm for nm in LABEL_CANDIDATES if nm in lower_to_orig), None)

if label_key is not None:
    lbl = lower_to_orig[label_key]
    y = map_attack_benign(df[lbl])
    X = df.drop(columns=[lbl], errors="ignore")
else:
    last = df.columns[-1]
    s = df[last].astype(str).str.strip().str.lower()
    if set(s.unique()) <= {"0","1","normal","benign","attack","malicious","true","false"}:
        y = map_attack_benign(df[last])
        X = df.drop(columns=[last], errors="ignore")
    else:
        print("[warn] No label-like column found; proceeding without labels.")
        y = None
        X = df.copy()

print("Label distribution (0=benign, 1=attack):")
if y is not None:
    print(y.value_counts())

# Timestamp normalization
ts_key = next((nm for nm in TS_CANDIDATES if nm in lower_to_orig), None)
if ts_key is not None:
    ts_col = lower_to_orig[ts_key]
    if pd.api.types.is_numeric_dtype(X[ts_col]):
        ts = pd.to_datetime(X[ts_col], unit="s", errors="coerce", utc=True)
    else:
        ts = pd.to_datetime(X[ts_col], errors="coerce", utc=True)
    X["ts_epoch"]   = ts.view("int64") // 10**9
    X["ts_hour"]    = ts.dt.hour
    X["ts_weekday"] = ts.dt.weekday
    X = X.drop(columns=[ts_col])

# Ratios
add_ratio(X, "spkts","dpkts","ratio_s_d_pkts")
add_ratio(X, "dpkts","spkts","ratio_d_s_pkts")
add_ratio(X, "sbytes","dbytes","ratio_s_d_bytes")
add_ratio(X, "dbytes","sbytes","ratio_d_s_bytes")

print("After cleaning + feature creation:", X.shape)
X.head(3)


In [ ]:

# ===== Memory-safe encoding + scaling =====
X_work = X.copy()

# Drop huge-cardinality identifiers (if present)
for bad in ("srcip","dstip"):
    if bad in X_work.columns:
        X_work = X_work.drop(columns=[bad])

# Convert object columns (except selected categoricals) to numeric where possible
for c in X_work.columns:
    if X_work[c].dtype == "object" and c not in FORCE_CAT:
        X_work[c] = pd.to_numeric(X_work[c], errors="ignore")

# Cap & one-hot only selected categoricals
cat_cols = [c for c in FORCE_CAT if c in X_work.columns]
for c in cat_cols:
    X_work[c] = cap_categories(X_work[c], k=TOPK_CATS)

X_enc = pd.get_dummies(X_work, columns=cat_cols, dummy_na=False)

# Downcast numerics to float32
num_cols = X_enc.select_dtypes(include=[np.number]).columns.tolist()
X_enc[num_cols] = X_enc[num_cols].apply(pd.to_numeric, errors="coerce", downcast="float").fillna(0)

# Scale numerics
if SCALER == "minmax":
    mn = X_enc[num_cols].min(axis=0).astype(np.float32)
    mx = X_enc[num_cols].max(axis=0).astype(np.float32)
    denom = (mx - mn).replace(0, 1)
    X_scaled = (X_enc[num_cols] - mn) / denom
else:
    mu = X_enc[num_cols].mean(axis=0).astype(np.float32)
    sd = X_enc[num_cols].std(axis=0, ddof=0).replace(0, 1)
    X_scaled = (X_enc[num_cols] - mu) / sd.astype(np.float32)

print("Encoded+Scaled shape:", X_scaled.shape)
print("Approx memory (MB):", round(X_scaled.memory_usage(deep=True).sum()/1e6, 1))
X_scaled.head(3)


In [ ]:

# ===== Summary statistics =====
summary = X_scaled.describe().transpose()
summary_path = OUT_DIR / "summary_stats.csv"
summary.to_csv(summary_path, index=True)
with open(OUT_DIR / "summary_stats.txt", "w") as f:
    f.write(f"Rows: {X_scaled.shape[0]}, Cols: {X_scaled.shape[1]}\n")
    f.write(summary.head(20).to_string())
print("Saved summary to:", summary_path)
summary.head(10)


In [ ]:

# ===== Correlation heatmap (sample up to 40 columns) =====
sample_cols = X_scaled.columns[:min(40, X_scaled.shape[1])]
corr_sample = X_scaled[sample_cols].corr(numeric_only=True)

plt.figure(figsize=(8,6))
plt.title("Correlation Heatmap (sample)")
plt.imshow(corr_sample.values, aspect='auto', interpolation='nearest')
plt.xticks(range(len(sample_cols)), sample_cols, rotation=90, fontsize=7)
plt.yticks(range(len(sample_cols)), sample_cols, fontsize=7)
plt.colorbar()
plt.tight_layout()
heatmap_path = OUT_DIR / "correlation_heatmap.png"
plt.savefig(heatmap_path, dpi=150)
plt.show()
print("Saved heatmap to:", heatmap_path)


In [ ]:

# ===== PCA via NumPy + scatter =====
try:
    X_np = X_scaled.to_numpy(dtype=float)
    X_pca, k = pca_numpy(X_np, keep_var=PCA_KEEP_VAR)
    print(f"PCA components kept: {k}")

    if X_pca.shape[1] >= 2:
        plt.figure(figsize=(7,6))
        plt.title("PCA Scatter (PC1 vs PC2)")
        plt.scatter(X_pca[:,0], X_pca[:,1], s=3, alpha=0.35)
        plt.xlabel("PC1"); plt.ylabel("PC2")
        plt.tight_layout()
        pca_path = OUT_DIR / "pca_scatter.png"
        plt.savefig(pca_path, dpi=150)
        plt.show()
        print("Saved PCA scatter to:", pca_path)
except Exception as e:
    print("[warn] PCA step failed:", e)


In [ ]:

# ===== Save cleaned CSV (features + label_bin if available) =====
cleaned = X_scaled.copy()
if 'y' in globals() and y is not None:
    cleaned["label_bin"] = y.values

cleaned_path = OUT_DIR / "unsw_nb15_cleaned.csv"
cleaned.to_csv(cleaned_path, index=False)
print("Saved cleaned dataset to:", cleaned_path)
cleaned.head(3)
